In [1]:
def eigen_finder_kth(Pi, k):
    eigenvals, eigenvecs = torch.linalg.eig(Pi)
    magnitudes = torch.abs(eigenvals)
    values, indices = torch.topk(magnitudes, k, largest=False)
    smallest_eigenvals = eigenvals[indices]
    smallest_eigenvecs = eigenvecs[:, indices]
    return smallest_eigenvecs, smallest_eigenvals

In [ ]:
def eigen_finder_abs(Pi, threshold):
    eigenvals, eigenvecs = torch.linalg.eig(Pi)
    magnitudes = torch.abs(eigenvals)
    zero_eigenvals = []
    zero_eigenvecs = []
    for i in range(len(eigenvals)):
        if magnitudes[i] <= threshold:
            zero_eigenvals.append(eigenvals[i])
            zero_eigenvecs.append(eigenvecs[i])
    return zero_eigenvecs, zero_eigenvals


In [ ]:
def eigen_finder_scaled(Pi, threshold):
    eigenvals, eigenvecs = torch.linalg.eig(Pi)
    magnitudes = torch.abs(eigenvals)
    total_mag = torch.sum(magnitudes)
    border = total_mag * threshold
    zero_eigenvals = []
    zero_eigenvecs = []
    for i in range(len(eigenvals)):
        if magnitudes[i] <= border:
            zero_eigenvals.append(eigenvals[i])
            zero_eigenvecs.append(eigenvecs[i])
    return zero_eigenvecs, zero_eigenvals


In [ ]:
def spin_up(x, model_parameters, model,spin_up_time):
    (N, dx, dt, alpha, beta, F_L96) = model_parameters 
    x_in = x

    for i in range(spin_up_time):
        x_out = model(x_in, model_parameters)
        # print(x_out.shape)
        x_in = x_out
    return x_out


def stencil_selector(grid_index, Ensemble, N, ensemble_size,stencil_members):

    i = grid_index
    # print(i)
    # print(stencil_members)

    subset = torch.zeros(len(stencil_members), ensemble_size)
    for ii in range(len(stencil_members)):
        j = grid_index + stencil_members[ii]
        # print(j)
        subset[(ii%N),:] = Ensemble[(j%N),:]
        

    return subset


def put_in_place(n_i, N, N_tilde, grid_index, stencil_members):
    # print(len(stencil_members))
    # print(n_i.shape)
    sub_row = torch.zeros(1,N)
    for i in range(len(stencil_members)):
        # print(i)
        local = grid_index + stencil_members[i]
        sub_row[1, local%N] = n_i[i]
    N_tilde = torch.stack(N_tilde, sub_row)
    return N_tilde

def little_GETLM(Chi_i, Xi_i, X_i, stencil_members_large):
    # Construct matrix
    # (1)
    Pi_half = torch.cat((Chi_i,-Xi_i, -X_i), dim=0)
    Pi = Pi_half @ Pi_half.T
    # print(Pi_half.shape)

    # (2)

    # 




    # (3)
    # Split into two 9-vectors
    split_1 = len(stencil_members_large[0])
    split_2 = len(stencil_members_large[1])
    n_i = smallest_eigenvecs[:split_1]
    l_i = smallest_eigenvecs[split_1, split_2]
    k_i = smallest_eigenvecs[split_2:]

    return n_i, l_i

def GETLM_generator(ensemble_size, sd, x_out, model_parameters, model, stencil_members_large):
    (N, dx, dt, alpha, beta, F_L96) = model_parameters 
    N_tilde = torch.zeros(1,N)
    L_tilde = torch.zeros(1,N)
    K_tilde = torch.zeros(1,N)

    future_members = stencil_members_large[0]
    current_members = stencil_members_large[1]
    past_members = stencil_members_large[1]

    X = torch.zeros(N, ensemble_size)
    Xi =  torch.zeros(N, ensemble_size)
    Chi = torch.zeros(N, ensemble_size)

    for i in tqdm(range(ensemble_size)):
        x_pert = torch.randn(N,1)*sd
        # print(x_pert)
        x_in_pert = x_out + x_pert
        x_out_pert = model(x_in_pert,model_parameters)
        x_out_unpert = model(x_out, model_parameters)
        Xi_i = x_out_pert - x_out_unpert
  
        x_out_out_pert = model(x_out_pert, model_parameters)
        x_out_out_unpert = model(x_out_unpert, model_parameters)

        Chi_i = x_out_out_pert - x_out_out_unpert


        X[:,i] = x_pert.squeeze()
        Xi[:,i] = Xi_i.squeeze()
        Chi[:,i] = Chi_i.squeeze()

    for grid_index in range(N):


        X_i = stencil_selector(grid_index, X, N, ensemble_size,past_members)
        Xi_i = stencil_selector(grid_index, X, N, ensemble_size,current_members)
        Chi_i = stencil_selector(grid_index, Chi, N, ensemble_size,future_members)

        n_i, l_i, k_i= little_GETLM(Chi_i, Xi_i, X_i, stencil_members_large)
        # print(n_i.shape)
        # print(l_i)

        # print(M_i.shape)
        # print(sum(abs(Chi_i - M_i @ X_i)))
        # print(sum(abs(Chi_i[:,0] - M_i @ X_i[:,0])))
        # x_pert_test = torch.randn(N,1)*sd
        # x_pert_test_t1 = model(x_pert_test,model_parameters)


        # print(sum(abs(Chi_i[:,0] - M_i @ X_i[:,0])))
        # print(sum(abs(x_pert_test_t1 - M_i @ x_pert_test)))

        # l_i = [1,2,3,4,5,6,7,8]


        N_tilde = put_in_place_row(n_i, N, N_tilde, grid_index, future_members)
        L_tilde = put_in_place_row(l_i, N, L_tilde, grid_index, current_members)
        K_tilde = put_in_place_row(k_i, N, K_tilde, grid_index, past_members)


    
    return  N_tilde, L_tilde

